# RAW to RGB — Part 1: Conversion and Normalization

In this notebook we:
1. Read a Canon RAW file as a raw Bayer array using `rawpy`
2. Determine the **black level** from a dark frame
3. Determine the **clipping point** from an overexposed frame
4. Normalize the image to the range 0…1

**Required files in the same folder as this notebook:**
- `IMG_2358.CR2` — black frame (lens cap on)
- `IMG_2364.CR2` — correctly exposed frame
- `IMG_2367.CR2` — overexposed frame (clipped whites)

In [ ]:
import numpy as np           # needed for image arrays
import rawpy                 # Needed for reading raw image files. Based on LibRAW
import os                    # Needed for file io
import HdM as HdM            # a useful function I prepared for you to display images pixel per pixel on your monitor
from scipy.ndimage import zoom  # function to scale images

## Monitor Nonlinearity according to the sRGB Standard
We define these once and reuse them throughout all notebooks.

$$\text{linear} \rightarrow \text{sRGB}: \quad
V = \begin{cases} 1.055\, L^{1/2.4} - 0.055 & L > 0.0031308 \\ 12.92\, L & \text{otherwise} \end{cases}$$

In [ ]:
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)
sRGB2linear = lambda x: np.where(x > 0.04045, ((x + 0.055) / 1.055) ** 2.4, x / 12.92)

## Load RAW files

We use `rawpy.imread(...).raw_image_visible` which gives us the
Bayer plane directly as a 2D uint16 array. We convert the uint16 values to float32 for further processing. But the values are still in 0...65535 range debending on the bit-depth of your camera. Let's find out!

In [ ]:
image_folder = ''   # adjust if your RAW files are in a different folder

def load_raw_bayer(path):
    """Return the visible Bayer plane as a float32 array (uint16 values)."""
    with rawpy.imread(path) as raw:
        return raw.raw_image_visible.astype(np.float32)

black_frame = load_raw_bayer(os.path.join(image_folder, 'RAW_FILENAME.EXTENSION')))  # Hinweis: Put in the correct filename for the RAW file of your black image.
raw_img     = load_raw_bayer(os.path.join(image_folder, 'RAW_FILENAME.EXTENSION')))  # Hinweis: Put in the correct filename for the RAW file of your correctly exposed image.
white_frame = load_raw_bayer(os.path.join(image_folder, 'RAW_FILENAME.EXTENSION')))  # Hinweis: Put in the correct filename for the RAW file of your overexposed white image.

print(f'Image shape : {raw_img.shape}  dtype: {raw_img.dtype}')
print(f'')

# Better check if you have loaded the correct images
print(f'\nThis is your black frame. Value range : {black_frame.min():.0f} … {black_frame.max():.0f}')
HdM.show(linear2sRGB(zoom(black_frame/65535,[1/16,1/16])))
print(f'\nThis is your correctly exposed Image. Value range : {raw_img.min():.0f} … {raw_img.max():.0f}')
HdM.show(linear2sRGB(zoom(raw_img/65535,[1/16,1/16])))
print(f'\nThis is your white frame. Value range : {white_frame.min():.0f} … {white_frame.max():.0f}')
HdM.show(linear2sRGB(zoom(white_frame/65535,[1/16,1/16])))

## First look — apply sRGB gamma so the image is visible

In [ ]:
HdM.show(linear2sRGB(raw_img / 65535))

## Determine the black level

The black frame (lens cap on) reveals the sensor's dark current and read noise.

In [ ]:
print(f'Black frame  min    : {np.min(black_frame):.1f}')
print(f'Black frame  mean   :')  # Hinweis: Calculate the mean of your RAW file
print(f'Black frame  median :')  # Hinweis: Calculate the median of your RAW file
print(f'Black frame  max    :')  # Hinweis: Calculate the max of your RAW file


In [ ]:

black_level =   # Hinweis: which method gives you the best estimation of your black level?
print(f'\nBlack level  = {black_level:.1f}')

## Measure read noise standard deviation

We crop away the border (optical black pixels) before measuring noise.
The standard deviation of the black frame tells us how much noise the sensor adds
even when no light falls on it.

In [ ]:
crop = 500
black_crop = black_frame[crop:-crop, crop:-crop]
black_noise_std = np.std(black_crop)
print(f'The read noise standard deviation is: {black_noise_std:.2f} code values')

# Visualize the black frame after subtracting the black level
HdM.show(linear2sRGB(np.clip((black_frame - black_level) / 50, 0, 1)))

In [ ]:
# What if we add a constant of three noise standard deviations before displaying the noise frame?
HdM.show(linear2sRGB(np.clip((black_frame - black_level + 3*black_noise_std) / 100, 0, 1)))
print('Notice there are only a few number of black "Holes" left. Bonus: How many out of 1000?')

## Determine the clipping point

The white - stongly overexposed frame shows us the maximum code value the sensor produces before clipping.

In [ ]:
print(f'Overexposed frame  min    : {np.min(white_frame):.1f}')
print(f'White frame  mean   :')  # Hinweis: Calculate the mean of your RAW file
print(f'White frame  median :')  # Hinweis: Calculate the median of your RAW file
print(f'White frame  max    :')  # Hinweis: Calculate the max of your RAW file
print(f'White frame  std    :')  # Hinweis: Calculate the standrad deviation of your RAW file

clipping_point =   # Hinweis: Which method gives you the best estimation of your clipping point?
print(f'\nClipping point = {clipping_point:.1f}')


In [ ]:
# Sometimes the clipping is uniform, sometimes there are artifacts to observe:
HdM.show(linear2sRGB(np.clip((white_frame - np.median(white_frame) + 0.5)/max(8,(white_frame.max() - np.median(white_frame)))*2, 0, 1)))

## Normalize to 0…1

We subtract the black level and divide by the dynamic range:

$$\hat{Image} = \frac{Image - \text{blackLevel}}{\text{clippingPoint} - \text{blackLevel}}$$

The `stops_above_white` parameter lets us adjust the mapping of white from the raw data to your display in powers of 2 (stops).

In [ ]:
raw_img_black_subtracted =   # Hinweis: Scale the image so that black level matches 0.0 and clipping_point matches 1.0

stops_above_white = 2   # try 1, 2, 3 and values inbetween until the white patch of the color checker renders white on the monitor

HdM.show(linear2sRGB(np.clip(zoom(raw_img_black_subtracted,[0.1,0.1]) * 2**stops_above_white, 0, 1)))

print("How many stops of overexposure delivers your camera? How many stops would be ideal? Does this depend on the application, e.g. Photo or Video?")

## Save result for the next notebook

In [ ]:
# Normalize black_noise_std to the same scale as the image
black_noise_std_norm = black_noise_std / (clipping_point - black_level)

np.savez('20_RawBayerImage_BlackSubtracted.npz',
         raw_img_black_subtracted = raw_img_black_subtracted,
         black_noise_std=black_noise_std_norm)

print('Saved: 20_RawBayerImage_BlackSubtracted.npz')